# TruePrice Local Fruit + Camel YOLO Training

Use this notebook in Google Colab with `dataset/trueprice_yolo_local.zip`.

Classes:

```text
0 apple
1 banana
2 grape
3 mango
4 strawberry
5 camel_doll
```

Important: fruit labels are weak centered boxes because the fruit source is a classification dataset. Direct camel photos are also weak centered boxes unless manually labeled later.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
!pip install -q ultralytics

## Upload Dataset ZIP

Upload `trueprice_yolo_local.zip` from your project path:

`/Users/shyoon840/HGU/3-1/HCI/TeamProject/hci_222/dataset/trueprice_yolo_local.zip`

In [ ]:
from google.colab import files

uploaded = files.upload()
ZIP_PATH = next(iter(uploaded.keys()))
ZIP_PATH

In [ ]:
from pathlib import Path
import shutil
import zipfile

WORKDIR = Path('/content/trueprice_yolo_local_work')
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(WORKDIR)

DATASET_DIR = WORKDIR / 'dataset' / 'trueprice_yolo_local'
DATA_YAML = DATASET_DIR / 'data.yaml'

assert DATASET_DIR.exists(), f'Dataset directory not found: {DATASET_DIR}'
assert DATA_YAML.exists(), f'data.yaml not found: {DATA_YAML}'

CLASS_NAMES = ['apple', 'banana', 'grape', 'mango', 'strawberry', 'camel_doll']
names = '[' + ', '.join(repr(name) for name in CLASS_NAMES) + ']'
DATA_YAML.write_text(
    f'''path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images

nc: {len(CLASS_NAMES)}
names: {names}
''',
    encoding='utf-8',
)

print(DATA_YAML.read_text())

In [ ]:
from collections import Counter

def count_images(path):
    return len([p for p in path.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])

counts = Counter()
split_counts = {}
for split in ['train', 'valid', 'test']:
    image_dir = DATASET_DIR / split / 'images'
    label_dir = DATASET_DIR / split / 'labels'
    split_counts[split] = count_images(image_dir)
    image_stems = {p.stem for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}}
    label_stems = {p.stem for p in label_dir.glob('*.txt')}
    assert image_stems == label_stems, f'{split}: image/label mismatch'
    for label_path in label_dir.glob('*.txt'):
        for line in label_path.read_text().splitlines():
            if line.strip():
                parts = line.split()
                assert len(parts) == 5, f'bad label line: {label_path}: {line}'
                class_id = int(parts[0])
                assert 0 <= class_id < len(CLASS_NAMES), f'bad class id: {class_id}'
                coords = [float(v) for v in parts[1:]]
                assert all(0 <= v <= 1 for v in coords), f'bad coords: {label_path}: {line}'
                counts[class_id] += 1

print('Images by split:', split_counts)
print('Labels by class:')
for class_id, name in enumerate(CLASS_NAMES):
    print(class_id, name, counts[class_id])

## Train

For this weak-label single-object dataset, start with conservative augmentation. Heavy mosaic/mixup can hurt because the intended input is one object per photo.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
print('device:', device)

model = YOLO('yolo11n.pt')
results = model.train(
    data=str(DATA_YAML),
    epochs=60,
    imgsz=640,
    batch=16,
    device=device,
    workers=2,
    seed=222,
    patience=15,
    project='/content/runs/trueprice',
    name='local_fruit_camel_yolo',
    exist_ok=True,
    close_mosaic=10,
    mosaic=0.15,
    mixup=0.0,
    degrees=5.0,
    translate=0.05,
    scale=0.25,
    fliplr=0.5,
)

In [ ]:
from ultralytics import YOLO

best_pt = Path('/content/runs/trueprice/local_fruit_camel_yolo/weights/best.pt')
assert best_pt.exists(), best_pt

best_model = YOLO(str(best_pt))
metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=640, device=device)
print(metrics)

In [ ]:
from google.colab import files

best_pt = Path('/content/runs/trueprice/local_fruit_camel_yolo/weights/best.pt')
last_pt = Path('/content/runs/trueprice/local_fruit_camel_yolo/weights/last.pt')
print('best:', best_pt, best_pt.stat().st_size)
print('last:', last_pt, last_pt.stat().st_size if last_pt.exists() else 'missing')
files.download(str(best_pt))

## Optional: Quick Prediction Check

Upload a few phone photos and confirm the class names before applying `best.pt` to the server.

In [ ]:
test_uploads = files.upload()
best_model = YOLO(str(best_pt))
for image_name in test_uploads.keys():
    pred = best_model.predict(image_name, imgsz=640, conf=0.25, save=True)
    print(image_name, pred[0].names)
    for box in pred[0].boxes:
        cls = int(box.cls.item())
        conf = float(box.conf.item())
        print(' ', CLASS_NAMES[cls], round(conf, 4))